Data Generation Notebook

In [1]:
import pandas as pd
import numpy as np

df9 = pd.read_csv("data/raw/ucl/online_retail_09_10.csv", encoding="utf-8-sig")
df10 = pd.read_csv("data/raw/ucl/online_retail_10_11.csv", encoding="utf-8-sig")

df9 = df9.dropna(subset=["CustomerID"])
df10 = df10.dropna(subset=["CustomerID"])

df9["is_cancellation"] = df9["InvoiceNo"].astype(str).str.startswith("C")
df10["is_cancellation"] = df10["InvoiceNo"].astype(str).str.startswith("C")

df9["line_value"] = df9["Quantity"] * df9["UnitPrice"]
df10["line_value"] = df10["Quantity"] * df10["UnitPrice"]

orders = pd.concat([df9, df10]).groupby("InvoiceNo").agg(
    customer_id=("CustomerID", "first"),
    order_value=("line_value", "sum"),
    n_items=("StockCode", "nunique"),
    total_qty=("Quantity", "sum"),
    invoice_date=("InvoiceDate", "first"),
    is_cancellation=("is_cancellation", "first"),
    country=("Country", "first"),
).reset_index()

print(orders.shape)
print(orders.head())

(44876, 8)
  InvoiceNo  customer_id  order_value  n_items  total_qty    invoice_date  \
0    489434      13085.0       505.30        8        166  12/1/2009 7:45   
1    489435      13085.0       145.80        4         60  12/1/2009 7:46   
2    489436      13078.0       630.33       19        193  12/1/2009 9:06   
3    489437      15362.0       310.75       23        145  12/1/2009 9:08   
4    489438      18102.0      2286.24       17        826  12/1/2009 9:24   

   is_cancellation         country  
0            False  United Kingdom  
1            False  United Kingdom  
2            False  United Kingdom  
3            False  United Kingdom  
4            False  United Kingdom  


In [2]:
orders["order_value_abs"] = orders["order_value"].abs()

'''Bucket using the absolute value cuz with cancellations, 
   the order value can be negative. We want to see if the absolute value of the order 
   has any effect on cancellation rates.  ''' 

orders["value_bucket"] = pd.qcut(orders["order_value_abs"], q=5, duplicates="drop")

print(orders.groupby("value_bucket")["is_cancellation"].mean())
print()
print(orders["value_bucket"].value_counts())  # sanity check bucket sizes are ~even

value_bucket
(-0.001, 54.6]        0.714238
(54.6, 179.35]        0.102841
(179.35, 311.43]      0.022061
(311.43, 508.85]      0.013148
(508.85, 168469.6]    0.027967
Name: is_cancellation, dtype: float64

value_bucket
(-0.001, 54.6]        8976
(54.6, 179.35]        8975
(179.35, 311.43]      8975
(311.43, 508.85]      8975
(508.85, 168469.6]    8975
Name: count, dtype: int64


 Real UK retail data says higher-value orders are safer (less likely to be cancelled), not riskier. The likely real-world reason: very low-value orders are disproportionately administrative/error rows (single-item test orders, near-free promotional line items, pricing mistakes) that get voided — not "customers casually cancel cheap stuff." It's still a real pattern in the data, just probably driven by a different mechanism than "expensive orders → buyer's remorse."

In [3]:
from scipy import stats

positive_orders = orders[orders["order_value"] > 0]["order_value"]

shape, loc, scale = stats.lognorm.fit(positive_orders, floc=0)
sigma = shape
mu = np.log(scale)

print(f"Fitted log-normal: mu={mu:.3f}, sigma={sigma:.3f}")
print(f"\nReal order value describe():\n{positive_orders.describe()}")

Fitted log-normal: mu=5.596, sigma=1.086

Real order value describe():
count     36969.000000
mean        479.954264
std        1374.990573
min           0.380000
25%         160.800000
50%         305.250000
75%         489.260000
max      168469.600000
Name: order_value, dtype: float64


Method. Real order values (aggregated to order level, cancellations excluded since they carry negative quantities) were fit to a log-normal distribution using scipy.stats.lognorm.fit.

Result. The fit produced mu = 5.596, sigma = 1.086. The real data showed a mean of £479.95 against a median of £305.25 — a substantial mean/median gap confirming a heavy right skew, consistent with a small number of large bulk orders pulling the average upward.

Currency/market adjustment. UCI's dataset reflects UK wholesale/gift retail — many buyers are small businesses purchasing in bulk for resale, not end consumers. A direct GBP→INR currency conversion would therefore import that wholesale skew into a dataset meant to represent Indian consumer e-commerce (fashion, beauty, home, electronics accessories), producing an unrealistically high median order value (~₹28,000 under straight conversion).

Decision. We kept the real fitted shape parameter (sigma = 1.086) — since skewness is a structural retail property independent of currency — but rescaled the location parameter to target a market-appropriate median order value of ₹1,200 for the synthetic dataset (mu = ln(1200) ≈ 7.09). This produces order values with a realistic Indian consumer e-commerce scale while inheriting a real, empirically observed degree of skew rather than an arbitrary one.

In [4]:
customer_order_counts = orders.groupby("customer_id").size()

print(f"Customer order count describe():\n{customer_order_counts.describe()}")
print(f"\nValue counts (top 15 order-count values):\n{customer_order_counts.value_counts().sort_index().head(15)}")

# What fraction of customers are "one-time" buyers?
one_time_frac = (customer_order_counts == 1).mean()
print(f"\nFraction of customers with exactly 1 order: {one_time_frac:.3f}")

Customer order count describe():
count    5942.000000
mean        7.552339
std        15.972262
min         1.000000
25%         2.000000
50%         4.000000
75%         8.000000
max       510.000000
dtype: float64

Value counts (top 15 order-count values):
1     1461
2      875
3      622
4      482
5      390
6      281
7      254
8      177
9      158
10     138
11     107
12     103
13      91
14      93
15      60
Name: count, dtype: int64

Fraction of customers with exactly 1 order: 0.246


Method. Real orders were grouped by customer ID and counts were summarized to characterize how often customers repeat-purchase.

Result. Real data showed a mean of 7.55 orders per customer against a median of 4 — again a heavy right skew (max observed: 510 orders from a single customer), and 24.6% of customers were one-time buyers.

Fitting approach. Unlike the order-value log-normal fit, the generator's customer-frequency mechanism (Zipf-weighted sampling across a customer pool) has no closed-form parameter fit. Instead, a small simulation sweep was run across candidate combinations of customer-pool size and Zipf shape parameter (a), comparing each combination's resulting one-time-buyer fraction and mean order count against the real anchors (one_time_frac ≈ 0.246, mean_orders ≈ 7.55), to identify the combination that best reproduces observed real-world customer behavior.

Why this matters downstream. is_first_time_buyer carries one of the larger injected risk weights in the return-risk formula. If the underlying customer population's repeat-purchase mix is unrealistic, the proportion of first-time-buyer orders in the synthetic dataset would be unrealistic even though the risk weight itself is sound — skewing the overall base return rate and the model's learned importance for this feature.

In [5]:
import numpy as np

def simulate_negbinom_orders(n_customers, n_orders, r, p, seed=42):
    rng = np.random.default_rng(seed)
    raw_counts = rng.negative_binomial(r, p, size=n_customers)
    raw_counts = raw_counts + 1
    return raw_counts

target_mean = 7.55
target_one_time_frac = 0.246

results = []
for r in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    for p in [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:
        counts = simulate_negbinom_orders(n_customers=2000, n_orders=5000, r=r, p=p)
        mean_o = counts.mean()
        frac = (counts == 1).mean()
        results.append((r, p, mean_o, frac))

results.sort(key=lambda x: abs(x[2]-target_mean) + abs(x[3]-target_one_time_frac)*10)
for r, p, mean_o, frac in results[:10]:
    print(f"r={r}, p={p}: mean={mean_o:.2f}, one_time_frac={frac:.3f}")

r=0.65, p=0.09: mean=7.56, one_time_frac=0.208
r=0.55, p=0.08: mean=7.38, one_time_frac=0.268
r=0.5, p=0.07: mean=7.36, one_time_frac=0.283
r=0.6, p=0.08: mean=7.91, one_time_frac=0.225
r=0.6, p=0.09: mean=6.93, one_time_frac=0.248
r=0.75, p=0.1: mean=7.58, one_time_frac=0.180
r=0.7, p=0.1: mean=7.34, one_time_frac=0.197
r=0.7, p=0.09: mean=7.90, one_time_frac=0.191
r=0.55, p=0.07: mean=8.35, one_time_frac=0.235
r=0.65, p=0.1: mean=6.58, one_time_frac=0.234
